# AgentCore Harness 시작하기

**AgentCore Harness**에 오신 것을 환영합니다. 프레임워크 설정, 오케스트레이션 코드 작성, 배포 과정 없이 한 번의 API 호출로 에이전트를 정의하고 실행할 수 있어 개발자가 더 빠르게 에이전트를 실험하고 출시할 수 있습니다.

| 정보 | 세부 내용 |
|---|---|
| 튜토리얼 | 시작하기 - 생성, 호출, ExecuteCommand |
| SDK | boto3 |
| 모델 | Claude Haiku 4.5 및 Claude Sonnet 4.6 (Bedrock) |

이 노트북에서는 다음 작업을 진행합니다.
1. IAM 실행 역할 생성
2. Harness 에이전트 생성
3. 프롬프트로 에이전트 호출
4. 에이전트 VM에서 명령 실행
5. 리소스 정리

### 1. 종속성 설치

In [ ]:
!uv pip install -qU -r ../requirements.txt

In [ ]:
!uv pip freeze | grep boto3

계속하기 전에 Jupyter 커널을 다시 시작하세요.

### 2. 설정

프로젝트 루트에서 공용 헬퍼를 가져옵니다. 이 헬퍼는 IAM 역할 생성, boto3 클라이언트 설정 및 수명 주기 작업을 처리합니다.

In [ ]:
import sys
import time
import uuid
from pathlib import Path
import boto3

# 헬퍼
sys.path.insert(0, str(Path.cwd().parent))

# --- 설정 ---
from helper.iam import create_harness_role, delete_harness_role
from helper.client import get_agentcore_control_client, get_agentcore_client

# --- boto3 클라이언트 생성 ---
control = get_agentcore_control_client()
client = get_agentcore_client()

account_id = boto3.client("sts").get_caller_identity()["Account"]
print(f"Account: {account_id}")

### 3. IAM 역할 생성

필요한 모든 권한이 포함된 IAM 실행 역할을 생성합니다. 이 작업은 멱등성을 보장하므로 역할이 이미 존재하면 기존 ARN을 반환합니다.

In [ ]:
role_arn = create_harness_role()
print(f"\nExecution Role ARN: {role_arn}")

print("Waiting for IAM role to propagate...")
time.sleep(10)
print("Ready!")

### 4. Harness 에이전트 생성

고유한 이름으로 새 Harness 에이전트를 생성합니다. 에이전트는 `CREATING` 상태로 시작하고 몇 초 후 `READY`로 전환됩니다.

In [ ]:
HARNESS_NAME = f"GettingStarted_{uuid.uuid4().hex[:8]}"

resp = control.create_harness(
    harnessName=HARNESS_NAME,
    executionRoleArn=role_arn,
)
harness = resp["harness"]
harness_id = harness["harnessId"]
harness_arn = harness["arn"]
print(f"Harness ID: {harness_id}")
print(f"Harness ARN: {harness_arn}")
print(f"Status: {harness['status']}")

# 준비 상태 확인
for i in range(12):
    resp = control.get_harness(harnessId=harness_id)
    status = resp["harness"]["status"]
    print(f"Attempt {i + 1}: {status}")
    if status == "READY":
        print("\u2705 Harness is ready")
        break
    time.sleep(5)

### 5. 에이전트 호출

에이전트에 메시지를 보냅니다. 호출할 때마다 격리된 microVM을 식별하는 `session_id`가 필요합니다. 응답은 스트리밍되며, 텍스트와 도구 호출이 이벤트로 전달됩니다.

> 💡 메시지나 모델을 변경하며 실험해 보세요. 에이전트에서는 기본적으로 `file_operations` 및 `shell` 도구를 사용할 수 있습니다.

In [ ]:
session_id = str(uuid.uuid4()).upper()
print(f"Session ID: {session_id}\n")

response = client.invoke_harness(
    harnessArn=harness_arn,
    runtimeSessionId=session_id,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": "What are three fun things to do in Seattle on a rainy day?"
                    "Save your answer to a Markdown file."
                }
            ],
        }
    ],
    model={"bedrockModelConfig": {"modelId": "global.anthropic.claude-haiku-4-5-20251001-v1:0"}},
)

for event in response["stream"]:
    if "contentBlockStart" in event:
        start = event["contentBlockStart"].get("start", {})
        if "toolUse" in start:
            print(f"\n[Tool: {start['toolUse'].get('name', '?')}]", flush=True)
    elif "contentBlockDelta" in event:
        delta = event["contentBlockDelta"].get("delta", {})
        if "text" in delta:
            print(delta["text"], end="", flush=True)
    elif "messageStop" in event:
        print()
    elif "internalServerException" in event:
        print(f"\nError: {event['internalServerException']}")

이제 같은 세션 ID를 재사용하면서 모델을 변경하고, 새 모델로 새 파일을 생성해 보겠습니다.

In [ ]:
print(f"Session ID: {session_id}\n")

response = client.invoke_harness(
    harnessArn=harness_arn,
    runtimeSessionId=session_id,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": "What are three fun things to do in Seattle on a rainy day?"
                    "Save your answer to a Markdown file with Sonnet prefix."
                }
            ],
        }
    ],
    model={"bedrockModelConfig": {"modelId": "global.anthropic.claude-sonnet-4-6"}},
)

for event in response["stream"]:
    if "contentBlockStart" in event:
        start = event["contentBlockStart"].get("start", {})
        if "toolUse" in start:
            print(f"\n[Tool: {start['toolUse'].get('name', '?')}]", flush=True)
    elif "contentBlockDelta" in event:
        delta = event["contentBlockDelta"].get("delta", {})
        if "text" in delta:
            print(delta["text"], end="", flush=True)
    elif "messageStop" in event:
        print()
    elif "internalServerException" in event:
        print(f"\nError: {event['internalServerException']}")

### 6. 에이전트 VM에서 명령 실행

모든 Harness 세션은 격리된 microVM에서 실행됩니다. `execute_command`를 사용하면 VM에서 셸 명령을 직접 실행할 수 있으며, 환경을 점검하거나 에이전트가 생성한 파일을 읽을 때 유용합니다.

> ⚠️ **왜 `!ls`를 사용하지 않나요?** Jupyter의 `!`는 Jupyter가 실행 중인 **로컬 환경**에서 명령을 실행합니다. `execute_command`는 AWS에 있는 **에이전트의 원격 microVM**에서 실행됩니다.

In [ ]:
def run(cmd: str):
    print(f"$ {cmd}")
    resp = client.invoke_agent_runtime_command(
        agentRuntimeArn=harness_arn,
        runtimeSessionId=session_id,
        body={"command": cmd},
    )
    for event in resp["stream"]:
        if "chunk" in event:
            chunk = event["chunk"]
            if "contentDelta" in chunk:
                d = chunk["contentDelta"]
                if "stdout" in d:
                    print(d["stdout"], end="", flush=True)
                if "stderr" in d:
                    print(d["stderr"], end="", flush=True)
            elif "contentStop" in chunk:
                print(f"\n[exit: {chunk['contentStop']['exitCode']}]")
    print()

In [ ]:
run("ls -la")

In [ ]:
run("pwd")

In [ ]:
# 파일 목록을 확인한 다음 각 .md 파일 읽기
run('for f in *.md; do echo "=== $f ==="; cat "$f"; echo; done')

### 7. 리소스 정리

작업을 마치면 Harness 에이전트와 IAM 역할을 삭제합니다.

In [ ]:
control.delete_harness(harnessId=harness_id)
print(f"Deleted harness: {harness_id}")

In [ ]:
delete_harness_role()